In [34]:
def entero_a_binario(n):
    if n < 0:
        raise ValueError("El número debe ser un entero no negativo")
    elif n == 0:
        return "0"
    
    binario = ""
    while n > 0:
        binario = str(n % 2) + binario
        n //= 2
    return binario

def fraccion_a_binario(f, max_bits=20):
    if f < 0 or f >= 1:
        raise ValueError("La fracción debe ser positiva y menor que 1")
    elif f == 0:
        return "0", "0"

    binario = "0."
    count = 0
    while f != 0 and count < max_bits:
        f_nuevo = f * 2
        binario = binario + str(int(f_nuevo))
        f = f_nuevo - int(f_nuevo) 
        count += 1
        max_bit_break = (f != 0)
    return binario, max_bit_break

def decimal_a_binario(x):
    binario_entero = entero_a_binario(int(x))
    binario_fraccion, max_bit_break = fraccion_a_binario(x - int(x))
    return binario_entero + binario_fraccion[1:], max_bit_break

# A. Aplicando estas funciones a los siguientes números en base 10:

num = [11.1875, 18.0308, 26.0985, 94.0516]

print('\nA:')

for x in num:
    print(f"  ({x})_10 equivale a: ({decimal_a_binario(x)[0]})_2. ¿Se alcanzó el máximo de bits permitido?: {decimal_a_binario(x)[1]}")

# B. En el caso de 0.3 en base 10:

print(f"\nB: \n  (0.3)_10 equivale a: ({decimal_a_binario(0.3)[0]})_2. ¿Se alcanzó el máximo de bits permitido?: {decimal_a_binario(0.3)[1]}")

# Se observa que el número alcanzó el máximo de 20 bits, mostrando una representación no exacta

# C. Para (0.1)_10 y (0.2)_10:

print(f"\nC: \n  (0.1)_10 equivale a: ({decimal_a_binario(0.1)[0]})_2. ¿Se alcanzó el máximo de bits permitido?: {decimal_a_binario(0.1)[1]}")
print(f"  (0.2)_10 equivale a: ({decimal_a_binario(0.2)[0]})_2. ¿Se alcanzó el máximo de bits permitido?: {decimal_a_binario(0.2)[1]}")

print("\n  Según Python, ¿0.1 + 0.2 = 0.3? :", 0.1 + 0.2 == 0.3)

# Como se observa en la línea anterior, la suma 0.1 + 0.2 no es igual a 0.3 en Python. La razón detrás de esto se observa en los cálculos 
# realizados por la función 'decimal_a_binario', en los cuales se observa que 0.1 y 0.2 no tiene una representación exacta en binario, siendo
# que continúan con cifrás más allá del máximo de 20 bits. Esto resulta en que la suma de estos dos realizada por Python tampoco devuelva un 
# resultado exacto, y por lo tanto no pueda considerarse totalmente igual al valor "0.3" por medio de los operadores == o !=.

# D. Para una función que siga el algoritmo IEEE 754 de precisión simple (o 32 bits)

def IEEE32 (x):   # Toma como argumento un número
    binario, max_bit_break = decimal_a_binario(abs(x)) # Usar la función anterior, la cual usa las primeras dos funciones por separado para calcular
                                                           # las partes entera y decimal por separado y unirlas.
    # Signo (1 bit):
    if x >= 0:
        signo = "0"
    elif x < 0:
        signo = "1"

    # Exponente:
    arg_sep = binario.find(".")  # Buscar la posición del separador. Si no se encuentra (el numero no tiene parte decimal), devolverá un "-1"
    arg_uno = binario.find("1")  # Buscar el primer 1, ayudará a determinar si el exponente es positivo o negativo.
                                 # Si el primer uno se encuentra después del separador, el exponente es negativo. De lo contrario es positivo
    if arg_sep == -1:
        arg_sep = len(binario)   # Si no hay ningun punto, este se encuentra al final del número, como "xxxx.0"

    if abs(x) >= 1:              # Si el numero tiene parte entera distinta de 0
        exp_real = arg_sep - 1   
    else:                        # Si el numero es mayor que -1 y menor que 1 
        exp_real = 1 - arg_uno 

    exponente = entero_a_binario(exp_real + 127) # Agregar las 127 unidades de bias y pasar la cifra a binario 

    if x ==0:
        exponente = "0"

    while len(exponente) < 8:
        exponente = "0" + exponente   # Agregar ceros por la izquierda hasta completar los 8 bits requeridos

    # Mantissa:
    mantissa = binario[arg_uno+1:].replace(".", "")  # Organizar la mantissa desde la cifra posterior al primer 1. Luego, quitar el separador

    if len(mantissa) > 23:
        mantissa = mantissa[:23]
    elif len(mantissa) < 23:
        while len(mantissa) < 23:
            mantissa = mantissa + "0"  # A la mantissa se le agregan ceros a la derecha, al contrario que con el exponente

    binario_entero = signo + exponente + mantissa

    return {'Signo' : signo, 
            'Exponente' : exponente, 
            'Mantissa' : mantissa, 
            'Binario entero' : binario_entero } 

# La función directa para este algoritmo es:

import struct

def IEEE32_struct(x):
    pck = struct.pack('>f', x) # 32-bit float
    bits = ''.join(f'{byte:08b}' for byte in pck)
    return bits

# Para la comparación usamos la lista "num":

print("\nD: \n  Los valores propios y de la función son:")

for x in num:
    binario_manual = IEEE32(x)
    binario_struct = IEEE32_struct(x)

    print(f"  > Para {x}: \n    Con la función propia: Signo: {binario_manual['Signo']}, Exponente: {binario_manual['Exponente']}, Mantissa: {binario_manual['Mantissa']}, Binario entero: {binario_manual['Binario entero']}")
    print(f"    Con struc.pack:        Signo: {binario_struct[0]}, Exponente: {binario_struct[1:9]}, Mantissa: {binario_struct[9:]}, Binario entero: {binario_struct}")
    print(f"    Comparando ambos números, binario_manual == binario_struct : {binario_manual["Binario entero"] == binario_struct}")

    



A:
  (11.1875)_10 equivale a: (1011.0011)_2. ¿Se alcanzó el máximo de bits permitido?: False
  (18.0308)_10 equivale a: (10010.00000111111000101000)_2. ¿Se alcanzó el máximo de bits permitido?: True
  (26.0985)_10 equivale a: (11010.00011001001101110100)_2. ¿Se alcanzó el máximo de bits permitido?: True
  (94.0516)_10 equivale a: (1011110.00001101001101011010)_2. ¿Se alcanzó el máximo de bits permitido?: True

B: 
  (0.3)_10 equivale a: (0.01001100110011001100)_2. ¿Se alcanzó el máximo de bits permitido?: True

C: 
  (0.1)_10 equivale a: (0.00011001100110011001)_2. ¿Se alcanzó el máximo de bits permitido?: True
  (0.2)_10 equivale a: (0.00110011001100110011)_2. ¿Se alcanzó el máximo de bits permitido?: True

  Según Python, ¿0.1 + 0.2 = 0.3? : False

D: 
  Los valores propios y de la función son:
  > Para 11.1875: 
    Con la función propia: Signo: 0, Exponente: 10000010, Mantissa: 01100110000000000000000, Binario entero: 01000001001100110000000000000000
    Con struc.pack:        Sig